# Project DAT305 - Motion Regognition from IMU data

By Andreas Nesse & Simen Kolbjørnsen

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

from src.functions import *
from src.classes import *

## Introduction:

1 step is approximately $500$ ms long on average. This means, with a $\Delta t$ of 0.0125 s or 12.5 ms between the IMU readings, 1 step is $500 / 12.5 = 40$ points. In each IMU reading we have 6 values, 3 for acceleration and 3 for gyroscope data.

By entering the data raw to the model, this means that the model has to take in 240 input values.

To compress this we can use the magnitude of the acceleration and gyroscope data, meaning that instead of 6 values per point, we get 2. Again leading to 80 input values for the model.

Furthermore it could be possible to just use statistical/mathematical properties of the data. Like computing the mean, sd, variance, max, min etc. This again could make the input size smaller than 20 values.

We have 2 627 548 data points from 75 different recording sessions from 31 different people. Meaning we have som variety in the data. This gives us the ability to train the model to hopefully be able to predict on data collected from people with varying step length, height, etc. Now since we are going to make windows of 40 points each, this leaves us with $2 627 548 / 40 = 65 688.7$ inputs of data to be used to train/validate/test. So just over $2^{16}$.

## 1. DATA
### 1.1 Loading the data

The data is stored in multiple csv files in the github repository [`SIMUL-dataset`](https://github.com/simpleLoc/SIMUL-dataset). We scraped the urls for the files and stored the relative urls in [`file_urls.csv`](./file_urls.csv), such that they can be added to the `BASE_URL` provided in the code block below to get the full path. 

We can set the `files_to_read` to the number of files we want to read. Our implementation will maximum read 75 files, as it is the amount of csv files that are stored in the repository.

In [ ]:
files_to_read = 75
max_files = 75
N = max(min(files_to_read, max_files), 1)
try:
    if N != N_loaded:
        raise NameError
    print(f"Data already loaded: {len(data)} files")
except NameError:
    BASE_URL = "https://raw.githubusercontent.com/simpleLoc/SIMUL-dataset/refs/heads/master/data/by-person/"
    
    file_URLs = pd.read_csv("https://raw.githubusercontent.com/andynesse/DAT305/refs/heads/main/file_urls.csv", header=None)[0]
    data = load_data(BASE_URL, file_URLs[:N])
    print(len(data))
N_loaded = N

Select and rename relevant parameters and then do describe on all the dataframes:

In [ ]:
columns = {'Timestamp' : 'ts', ' Activity' : 'activity', ' AccelX_3' : 'aX', ' AccelY_3' : 'aY', ' AccelZ_3' : 'aZ', ' GyroX_3' : 'gX', ' GyroY_3' : 'gY', ' GyroZ_3' : 'gZ'}
for i, df in enumerate(data):
    df = df.rename(columns=columns)
    df = df[list(columns.values())]
    print(df.describe())
    data[i] = df


Check if we have null values (Fortunately we have none for every column in every file):

In [ ]:

print(sum([df.isnull().sum() for df in data]))

Here we plot the magnetude of the acc and gyro for a dataframe:

In [ ]:
data_sample = data[0]

ts = data_sample['ts'].to_numpy()
aX = data_sample['aX'].to_numpy()
aY = data_sample['aY'].to_numpy()
aZ = data_sample['aZ'].to_numpy()
gX = data_sample['gX'].to_numpy()
gY = data_sample['gY'].to_numpy()
gZ = data_sample['gZ'].to_numpy()

plt.plot(ts, magnitude(aX, aY, aZ), label="Acceleration")
plt.plot(ts, magnitude(gX, gY, gZ), label="Gyro")

#plt.xlim(left = 155, right = 160)
plt.legend()
plt.show()



Piechart of how many of the datapoints are walking or still in the df:

In [ ]:
activities = data_sample['activity']
still = activities[activities == 0]
pie = [len(still)/len(activities), 1 - len(still)/len(activities)]
plt.pie(pie, labels=["still", "in motion"],colors=("black","pink"))
plt.show()

A closer look on the magnetude of acc and gyro, so that we can see the repeating signal of walking motion, here we see a 3 second snapshot. Since one step is on average 500ms, this should show around 6 steps:

In [ ]:
data_sample = data[0]

ts = data_sample['ts'].to_numpy()
aX = data_sample['aX'].to_numpy()
aY = data_sample['aY'].to_numpy()
aZ = data_sample['aZ'].to_numpy()
gX = data_sample['gX'].to_numpy()
gY = data_sample['gY'].to_numpy()
gZ = data_sample['gZ'].to_numpy()

plt.plot(ts, magnitude(aX, aY, aZ), label="Acceleration")
plt.plot(ts, magnitude(gX, gY, gZ), label="Gyro")
dt = 3
tx = 106
plt.xlim(left = tx, right = tx+dt)
plt.legend()
plt.show()

From the plot above we can see the spikes in acceleration and gyroscope data when steps are taken.

The amount of datapoints we have, combined from all the files:

In [ ]:
total_datapoints = sum(len(d) for d in data)
print(f"Total datapoints: {total_datapoints}")

We want to split our data into training, validation and test and we wanted a standard 70, 15, 15 split. Since we have 75 files we get:

$$75 \cdot 0.7 = 52.5 \approx 53$$
$$75 \cdot 0.15 = 11.25 \approx 11$$
$$53 + 11 + 11 = 75$$

The data is then splitted by using the above values for indexes, as data is just a list of dataframes.

In [ ]:
training_data = data[:53]
validation_data = data[53 : 53 + 11]
test_data = data[53 + 11:]
print(f"""
Files in each after splitted data:
Training: {len(training_data)}
Validation: {len(validation_data)}
Test: {len(test_data)}
""")

The reason for splitting the data as such is so when we are training the models, the validation and test data will contain unrelated data to the training data. It would be related if we had split the data randomly between files, because then points from the same walking session and same person would be in all the three splitted data. By splitting it before combining files, we achive that the data in validation and testing is completely unrelated and unseen by the training data.

To create windows for the data, we made the function [`to_windows`](./src/functions/to_windows.py#L9). It uses one of three window methods:

- [`window_by_parameters`](./src/functions/to_windows.py#L20): Create windows from all values from its columns. Resulting in data being size 40*6 = 240 for input. 
- [`window_by_magnitude`](./src/functions/to_windows.py#L33): Creates windows from the magnitude of the accelerometer data and the gyroscope data for each point. So the resulting size of the data here is 40*2 = 80.
- [`window_by_values`](./src/functions/to_windows.py#L43): Creates windows from statistical properties of the magnitudes. So the resulting size of the data here is 2*7 = 14.

We implemented these methods very generally, so that we easily can swap the number of columns we want to train on.

Our raw data also have 6 different activities:
- standing (0)
- walking (1)
- stairs up (2)
- stairs down(3)
- elevator up (4)
- elevator down (5)

We only want walking and not walking, so we converted activities in [`to_windows.py`](./src/functions/to_windows.py#L7) such that:
- Not walking (0): standing (0), elevator up (4), elevator down (5)
- Walking (1): walking(1), stairs up (2), stairs down (3)

In this way we only have two different output combinations, and thus the model we are going to train becomes much more simpler.

When creating the windoes we want the points in the window to be continous in terms of time, but we also want the points to be of the same label: walking (1) or not walking (0). To achieve this we did the most simple solution, to just drop the windows that are of two different activity labels.

Below we create windows for the training, validation and test data using the three different methods:

#### windows_by_parameters:

In [ ]:
windows_all_training = dfs_to_windows(training_data, window_by_parameters, 'activity', ['aX', 'aY', 'aZ', 'gX', 'gY', 'gZ'])
windows_all_training.describe()

In [ ]:
windows_all_validation = dfs_to_windows(validation_data, window_by_parameters, 'activity', ['aX', 'aY', 'aZ', 'gX', 'gY', 'gZ'])
windows_all_validation.describe()

In [ ]:
windows_all_test = dfs_to_windows(test_data, window_by_parameters, 'activity', ['aX', 'aY', 'aZ', 'gX', 'gY', 'gZ'])
windows_all_test.describe()

#### windows_by_magnitude:

In [ ]:
windows_magnitudes_training = dfs_to_windows(training_data, window_by_magnitude, 'activity', [("acc", 'aX', 'aY', 'aZ'), ("gyro", 'gX', 'gY', 'gZ')])
windows_magnitudes_training.describe()

In [ ]:
windows_magnitudes_validation = dfs_to_windows(validation_data, window_by_magnitude, 'activity', [("acc", 'aX', 'aY', 'aZ'), ("gyro", 'gX', 'gY', 'gZ')])
windows_magnitudes_validation.describe()

In [ ]:
windows_magnitudes_test = dfs_to_windows(test_data, window_by_magnitude, 'activity', [("acc", 'aX', 'aY', 'aZ'), ("gyro", 'gX', 'gY', 'gZ')])
windows_magnitudes_test.describe()

#### windows_by_value:

In [ ]:
windows_values_training = dfs_to_windows(training_data, window_by_values, 'activity', [("acc", 'aX', 'aY', 'aZ'), ("gyro", 'gX', 'gY', 'gZ')])
windows_values_training.describe()

In [ ]:
windows_values_validation = dfs_to_windows(validation_data, window_by_values, 'activity', [("acc", 'aX', 'aY', 'aZ'), ("gyro", 'gX', 'gY', 'gZ')])
windows_values_validation.describe()

In [ ]:
windows_values_test = dfs_to_windows(test_data, window_by_values, 'activity', [("acc", 'aX', 'aY', 'aZ'), ("gyro", 'gX', 'gY', 'gZ')])
windows_values_test.describe()

Our implementation discards the same amount of datapoints regardless of method used for windowing
The amount of datapoints discarded during the windowing process:

In [ ]:
discarded_datapoints = total_datapoints - 40 * (
    len(windows_all_training) + len(windows_all_validation) + len(windows_all_test)
)
print(f"Discarded datapoints after windowing: {discarded_datapoints}")
print(f"Fraction of datapoints that were discarded: {discarded_datapoints / total_datapoints * 100:.4}%")

To keep the different window data separated we made a class [`TrainingData`](./src/classes/training_data.py).

In [ ]:
all_points_data = TrainingData(
    windows_all_training,
    windows_all_validation,
    windows_all_test,
    'activity'
)

magnitude_data = TrainingData(
    windows_magnitudes_training,
    windows_magnitudes_validation,
    windows_magnitudes_test,
    'activity'
)

statistical_data = TrainingData(
    windows_values_training,
    windows_values_validation,
    windows_values_test,
    'activity'
)

## Classification model

Now we are going to do fit classification models to our test data using the 3 different wondowing approaches:

In [ ]:
all_points_classification_model = classification_model(all_points_data.n_cols)
all_points_classification_history = all_points_data.fit(all_points_classification_model, early_stopping=True)

In [ ]:
magnitude_classification_model = classification_model(magnitude_data.n_cols)
magnitude_classification_history = magnitude_data.fit(magnitude_classification_model, early_stopping=True)

In [ ]:
statistical_classification_model = classification_model(statistical_data.n_cols)
statistical_classification_history = statistical_data.fit(statistical_classification_model, early_stopping=True)

We plot the history of the models training:

In [ ]:
pd.DataFrame(all_points_classification_history.history).plot(figsize=(8, 5), grid=True)
pd.DataFrame(magnitude_classification_history.history).plot(figsize=(8, 5), grid=True)
pd.DataFrame(statistical_classification_history.history).plot(figsize=(8, 5), grid=True)
plt.show()

We can clearly see that the all_points_classification_model did worse in terms of val_loss than the two others. 

And the history showing loss and validation loss when training the models:

In [ ]:
pd.DataFrame(all_points_classification_history.history)

In [ ]:
pd.DataFrame(magnitude_classification_history.history)

In [ ]:
pd.DataFrame(statistical_classification_history.history)

### Evaluation and summary

In [ ]:
print(all_points_data.evaluate(all_points_classification_model))
print(magnitude_data.evaluate(magnitude_classification_model))
print(statistical_data.evaluate(statistical_classification_model))

In [ ]:
all_points_test = all_points_data.test_targets.to_numpy().ravel()
all_points_pred = (all_points_classification_model.predict(all_points_data.test_predictors, verbose=0).ravel() >= 0.5).astype(int)

print("Classification report for all_points:")
print(classification_report(
    all_points_test,
    all_points_pred,
    target_names=["not walking", "walking"],
    digits=4
))

print("Confusion matrix for all_points:")
print(confusion_matrix(
    all_points_test,
    all_points_pred,
    labels=[0,1]
))


In [ ]:
magnitude_test = magnitude_data.test_targets.to_numpy().ravel()
magnitude_pred = (magnitude_classification_model.predict(magnitude_data.test_predictors, verbose=0).ravel() >= 0.5).astype(int)

print("Classification report for magnitudes:")
print(classification_report(
    magnitude_test,
    magnitude_pred,
    target_names=["not walking", "walking"],
    digits=4
))

print("Confusion matrix for magnitudes:")
print(confusion_matrix(
    magnitude_test,
    magnitude_pred,
    labels=[0,1]
))


In [ ]:
statistical_test = statistical_data.test_targets.to_numpy().ravel()
statistical_pred = (statistical_classification_model.predict(statistical_data.test_predictors, verbose=0).ravel() >= 0.5).astype(int)

print("Classification report for statistical:")
print(classification_report(
    statistical_test,
    statistical_pred,
    target_names=["not walking", "walking"],
    digits=4
))

print("Confusion matrix for statistical:")
print(confusion_matrix(
    statistical_test,
    statistical_pred,
    labels=[0,1]
))

From the ouputs above we see the classification reports and confusion matrices for the different models. We also ran evaluate on the models before that.

The model using the magnitude approach performed best in our models here, with 99.31% test accuracy and 0.988 macro F1 score. Since the test set is imbalanced and contains 1619 windows that are "not walking" and 7717 that are "walking", accuracy may have been overstating the performance due to that inbalance here.

The confusion matrix also shows that the model predicted 44 "not walking" windows as "walking", and 20 "walking" windows as "not walking". So compared to 1619 "not walking" and 7717 "walking windows, the fraction $44/1619 = 0.027$ of non walking being predicted as walking is around 10 times higher error rate than the fraction $20/7717 = 0.0026$ of walking windows being predicted as non walking. Now if this differnce matter depends a lot on the use case, which we won't go deep into here. However for our usecase it would be better to falsly predict walking, than falsly predict non walking.

In [ ]:
print(f"Windows in test data: {len(magnitude_test)}")
print(f"Walking windows in test data: {len(magnitude_test[magnitude_test == 1]) / len(magnitude_test):.4}")
print(f"Non-walking windows in test data: {len(magnitude_test[magnitude_test == 0]) / len(magnitude_test):.4}")

If we were to just to predict walking every time we would have an accuracy of 82.66% on the test set. Now this is significantly worse than our accuracy of 99.31% with the magnitude model.

It is also worth mentioning that the model using the statistical window method for its data was very close in accuracy as the magnitude model, with an accuracy of 99.23%. That is just 0.08% worse than the magnitude model. Now these windows was very similar, as they both are using the magnitudes of the accelerometer and gyroscope data for the data points, but the statistical model uses smaller windows as they just stores statistical values such as the mean, std, min and max of the magnitudes.